# Notebook 05b — Restaurant Song Recommender (v2)

Rebuilds the recommender from `05_trialrecommender.ipynb` with three fixes that came out of reviewing that notebook:

1. **Consistent scale.** Instead of hand-typing abstract Spotify-PC combinations (which lands on an arbitrary, ungrounded scale), we author intent in real audio-feature terms (danceability, energy, etc.) and project it through the *actual* fitted `scaler_s` + `pca_s` from notebook 04 — the same pipeline the real song PCA scores went through. Every archetype target ends up on the same real scale, comparable to every other column of `W`.
2. **Full coverage.** `W` now covers all 8 Yelp PCs that have a clear, human-interpretable dominant loading (pc1–pc8), instead of silently zeroing out pc7/pc8.
3. **Corrected archetype meaning.** The original notebook's `X_train` comments labeled yPC4 as "Late-Night Dive" — but the actual loadings show yPC4's dominant signal is `NoiseLevel` (+0.50) and `stars` (-0.44), i.e. "loud/chaotic, lower-rated," not late-night. The true late-night axis is yPC7 (`GoodForMeal.latenight` +0.63 dominant). This notebook uses the loadings-verified labels, not the earlier guess.

This is still a hand-authored baseline, not a model fit to real listening data — the intent is that real user feedback (thumbs up/down) eventually replaces or corrects these targets.

In [16]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

PROCESSED = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\processed"

songs       = pd.read_csv(os.path.join(PROCESSED, 'song_pca.csv'))
restaurants = pd.read_csv(os.path.join(PROCESSED, 'restaurant_pca.csv'))

pca_spotify    = joblib.load(os.path.join(PROCESSED, 'spotify_pca.pkl'))
scaler_spotify = joblib.load(os.path.join(PROCESSED, 'spotify_scaler.pkl'))
pca_yelp       = joblib.load(os.path.join(PROCESSED, 'yelp_pca.pkl'))

print(f"Songs:       {songs.shape}")
print(f"Restaurants: {restaurants.shape}")

Songs:       (566780, 9)
Restaurants: (34808, 15)


---
## Step 1 — Separate identifiers from PC scores

In [17]:
SPOTIFY_PC_COLS = [c for c in songs.columns       if c.startswith('pc')]
YELP_PC_COLS    = [c for c in restaurants.columns if c.startswith('pc')]

song_vecs       = songs[SPOTIFY_PC_COLS].values
restaurant_vecs = restaurants[YELP_PC_COLS].values

N_SPOTIFY = len(SPOTIFY_PC_COLS)
N_YELP    = len(YELP_PC_COLS)

print(f"Song vectors:       {song_vecs.shape}")
print(f"Restaurant vectors: {restaurant_vecs.shape}")
print(f"Spotify PCs: {SPOTIFY_PC_COLS}")
print(f"Yelp PCs:    {YELP_PC_COLS}")

Song vectors:       (566780, 6)
Restaurant vectors: (34808, 13)
Spotify PCs: ['pc1', 'pc2', 'pc3', 'pc4', 'pc5', 'pc6']
Yelp PCs:    ['pc1', 'pc2', 'pc3', 'pc4', 'pc5', 'pc6', 'pc7', 'pc8', 'pc9', 'pc10', 'pc11', 'pc12', 'pc13']


---
## Step 1b — Deduplicate exact repeat tracks

`tracks_features.csv` has real duplicate rows: the same song (identical `name` + `artists`) appears under multiple Spotify track IDs, presumably from showing up on more than one release/album. Found while testing this notebook — one archetype's own top-3 recommended the same song twice ("Swervin' (feat. Diplo)", present under 4 different IDs). Left in, this would also distort any future hub/in-degree counting, since a duplicated song would look several times more "popular" than it really is just from row duplication. Dropping duplicates here, keeping the first occurrence of each `(name, artists)` pair.

In [18]:
before = len(songs)
songs = songs.drop_duplicates(subset=['name', 'artists']).reset_index(drop=True)
song_vecs = songs[SPOTIFY_PC_COLS].values

print(f"Dropped {before - len(songs)} duplicate (name, artists) rows ({(before - len(songs)) / before:.1%})")
print(f"Songs after dedup: {songs.shape}")

Dropped 23461 duplicate (name, artists) rows (4.1%)
Songs after dedup: (543319, 9)


---
## Step 2 — Read the PCA loadings

Same as notebook 05 — needed to verify what each PC actually represents before authoring any targets.

In [19]:
AUDIO_FEATURES = [
    'danceability', 'energy', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'loudness', 'tempo'
]

YELP_FEATURES = [
    'Ambience.romantic', 'Ambience.divey', 'Ambience.classy',
    'Ambience.hipster', 'Ambience.trendy', 'Ambience.upscale', 'Ambience.casual',
    'HasTV', 'HappyHour', 'RestaurantsGoodForGroups',
    'GoodForMeal.breakfast', 'GoodForMeal.brunch',
    'GoodForMeal.latenight', 'GoodForMeal.dinner',
    'RestaurantsTableService', 'NoiseLevel', 'stars'
]

n_yelp_feats    = pca_yelp.n_features_in_
n_spotify_feats = pca_spotify.n_features_in_
YELP_FEATURES   = YELP_FEATURES[:n_yelp_feats]
AUDIO_FEATURES  = AUDIO_FEATURES[:n_spotify_feats]

spotify_loadings = pd.DataFrame(
    pca_spotify.components_, index=SPOTIFY_PC_COLS, columns=AUDIO_FEATURES
).round(2)

yelp_loadings = pd.DataFrame(
    pca_yelp.components_, index=YELP_PC_COLS, columns=YELP_FEATURES
).round(2)

print("=== SPOTIFY loadings ===")
print(spotify_loadings.to_string())
print("\n=== YELP loadings (pc1-pc8) ===")
print(yelp_loadings.loc[['pc1','pc2','pc3','pc4','pc5','pc6','pc7','pc8']].to_string())

=== SPOTIFY loadings ===
     danceability  energy  speechiness  acousticness  instrumentalness  liveness  valence  loudness  tempo
pc1          0.30    0.48         0.16         -0.43             -0.28      0.13     0.32      0.48   0.19
pc2          0.55   -0.29         0.29          0.33             -0.33     -0.20     0.42     -0.16  -0.27
pc3         -0.19   -0.01         0.58          0.08             -0.17      0.71    -0.18     -0.09  -0.20
pc4         -0.01   -0.14         0.31          0.18              0.03     -0.01     0.09     -0.18   0.90
pc5          0.11    0.12         0.64         -0.26              0.36     -0.50    -0.33     -0.01  -0.13
pc6          0.26    0.08        -0.03         -0.00              0.78      0.35     0.42     -0.14  -0.06

=== YELP loadings (pc1-pc8) ===
     Ambience.romantic  Ambience.divey  Ambience.classy  Ambience.hipster  Ambience.trendy  Ambience.upscale  Ambience.casual  HasTV  HappyHour  RestaurantsGoodForGroups  GoodForMeal.breakfast 

---
## Step 3 — Human labels (loadings-verified)

`pc4` and `pc7` are corrected here versus the original notebook: `pc4`'s dominant loading is `NoiseLevel` (+0.50) / `stars` (-0.44), not late-night. `pc7`'s dominant loading is `GoodForMeal.latenight` (+0.63) — that's the real late-night axis.

In [20]:
YELP_PC_LABELS = {
    'pc1': 'sit-down dinner (table service + casual + happy hour)',
    'pc2': 'brunch / breakfast',
    'pc3': 'fine dining (upscale + classy + romantic, not casual)',
    'pc4': 'loud / chaotic (high noise, lower stars)',
    'pc5': 'hipster / trendy',
    'pc6': 'dive bar (very divey, some late-night overlap)',
    'pc7': 'late-night',
    'pc8': 'good for groups (divey/hipster-leaning, no table service)',
}

SPOTIFY_PC_LABELS = {
    'pc1': 'energetic + loud + danceable',
    'pc2': 'happy + danceable (acoustic-leaning)',
    'pc3': 'live performance + speechy',
    'pc4': 'fast tempo',
    'pc5': 'spoken word / pure instrumental',
    'pc6': 'live instrumental / jazz-like',
}

for pc, label in YELP_PC_LABELS.items():
    print(f"  {pc}: {label}")

  pc1: sit-down dinner (table service + casual + happy hour)
  pc2: brunch / breakfast
  pc3: fine dining (upscale + classy + romantic, not casual)
  pc4: loud / chaotic (high noise, lower stars)
  pc5: hipster / trendy
  pc6: dive bar (very divey, some late-night overlap)
  pc7: late-night
  pc8: good for groups (divey/hipster-leaning, no table service)


---
## Step 4 — Project audio-feature intent into Spotify-PC space

The Spotify PCA was fit on `log1p(SKEWED features) -> StandardScaler -> PCA` (notebook 04, Part 1). To land a hand-authored target on the same scale as real songs, we run it through the exact same pipeline instead of guessing PC combinations directly.

In [21]:
SKEWED = ['instrumentalness', 'acousticness', 'speechiness', 'liveness']

def audio_to_spotify_pc(audio_dict):
    """Project a raw audio-feature intent (real Spotify feature scale) into Spotify PC space."""
    row = pd.DataFrame([[audio_dict[f] for f in AUDIO_FEATURES]], columns=AUDIO_FEATURES)
    row[SKEWED] = row[SKEWED].apply(np.log1p)
    scaled = scaler_spotify.transform(row)
    return pca_spotify.transform(scaled)[0]

# sanity check against the real song PC distribution
print(songs[SPOTIFY_PC_COLS].agg(['mean', 'std']).round(2).to_string())

       pc1   pc2   pc3   pc4   pc5   pc6
mean -0.03 -0.00  0.00  0.00  0.00  0.01
std   1.81  1.15  1.06  0.94  0.89  0.87


---
## Step 5 — Author archetype intents (audio-feature space) and build W

One row per Yelp PC we're covering (pc1–pc8). Each intent is a hand-authored *hypothesis* grounded in the dominant loadings from Step 3 — still a guess, but now expressed in a space a human can actually reason about, and projected consistently.

**Anchored to the real dataset average, not just "reasonable-sounding" numbers.** An earlier draft of this cell picked plausible-looking values per archetype independently, and every one of them ended up below the dataset's average tempo and speechiness by accident — not because any archetype was deliberately meant to be slow/non-verbal, just because those happened to be the numbers that felt natural to type. That accidental shared bias swamped the real per-archetype signal (every archetype's "dominant Spotify PC" came out the same). Fix: start every archetype at the dataset mean (`scaler_spotify.mean_`, back-transformed out of log1p-space for the skewed features) and only move a feature away from that baseline when there's a specific, deliberate reason tied to that archetype's label.

**These numbers are still a starting hypothesis, not a verified answer** — they should be revisited once real user feedback exists.

In [22]:
# dataset averages (scaler_spotify.mean_ is in log1p-space for the 4 skewed features)
_mean_raw = scaler_spotify.mean_.copy()
for _f in SKEWED:
    _i = AUDIO_FEATURES.index(_f)
    _mean_raw[_i] = np.expm1(_mean_raw[_i])
NEUTRAL = dict(zip(AUDIO_FEATURES, _mean_raw))
print("Dataset-average ('neutral') audio profile:")
for f, v in NEUTRAL.items():
    print(f"  {f:18s} {v:.3f}")

Dataset-average ('neutral') audio profile:
  danceability       0.505
  energy             0.549
  speechiness        0.085
  acousticness       0.343
  instrumentalness   0.234
  liveness           0.189
  valence            0.410
  loudness           -10.718
  tempo              119.094


In [23]:
def archetype(**deviations):
    """Start from the dataset-average profile, override only the features this archetype deliberately differs on."""
    intent = dict(NEUTRAL)
    intent.update(deviations)
    return intent

AUDIO_INTENTS = {
    # pc1: sit-down dinner - close to neutral, slightly warmer and more acoustic
    'pc1': archetype(acousticness=0.40, valence=0.50, tempo=110.0),
    # pc2: brunch/breakfast - bright, upbeat, acoustic-leaning
    'pc2': archetype(danceability=0.60, acousticness=0.45, instrumentalness=0.15,
                      liveness=0.15, valence=0.75),
    # pc3: fine dining - deliberately quiet, slow, heavily instrumental/acoustic
    'pc3': archetype(danceability=0.30, energy=0.25, acousticness=0.65, instrumentalness=0.55,
                      liveness=0.15, loudness=-15.0, tempo=85.0),
    # pc4: loud/chaotic, lower-rated - deliberately high energy, fast, less refined
    'pc4': archetype(energy=0.80, speechiness=0.12, acousticness=0.15, instrumentalness=0.05,
                      liveness=0.35, loudness=-5.0, tempo=130.0),
    # pc5: hipster/trendy - close to neutral, slightly more upbeat/live
    'pc5': archetype(danceability=0.55, liveness=0.25, valence=0.50),
    # pc6: dive bar - gritty, raw, deliberately high liveness (live-band feel)
    'pc6': archetype(energy=0.70, acousticness=0.25, instrumentalness=0.15,
                      liveness=0.45, loudness=-7.0),
    # pc7: late-night - moody, dark, deliberately lower valence/energy/tempo, more instrumental
    'pc7': archetype(danceability=0.40, energy=0.35, acousticness=0.40, instrumentalness=0.40,
                      valence=0.30, loudness=-12.0, tempo=95.0),
    # pc8: good for groups - deliberately upbeat and danceable, sociable
    'pc8': archetype(danceability=0.70, energy=0.65, liveness=0.25, valence=0.65, loudness=-8.0),
}

ARCHETYPE_PCS = list(AUDIO_INTENTS.keys())          # ['pc1', ..., 'pc8']
n_archetypes  = len(ARCHETYPE_PCS)

X_train = np.zeros((n_archetypes, N_YELP))
Y_train = np.zeros((n_archetypes, N_SPOTIFY))

for row_i, pc in enumerate(ARCHETYPE_PCS):
    col_i = YELP_PC_COLS.index(pc)
    X_train[row_i, col_i] = 1.0
    Y_train[row_i] = audio_to_spotify_pc(AUDIO_INTENTS[pc])

model = LinearRegression(fit_intercept=False)
model.fit(X_train, Y_train)
W = model.coef_   # shape (N_SPOTIFY, N_YELP)

W_df = pd.DataFrame(W, index=SPOTIFY_PC_COLS, columns=YELP_PC_COLS)
print("Weight matrix W (rows=Spotify PCs, columns=Yelp PCs):")
print(W_df.round(3).to_string())

Weight matrix W (rows=Spotify PCs, columns=Yelp PCs):
       pc1    pc2    pc3    pc4    pc5    pc6    pc7    pc8  pc9  pc10  pc11  pc12  pc13
pc1 -0.013  0.482 -1.958  1.515  0.233  0.923 -1.079  1.026  0.0   0.0   0.0   0.0   0.0
pc2  0.277  1.051  0.160 -0.589  0.195 -0.558 -0.128  0.697  0.0   0.0   0.0   0.0   0.0
pc3  0.010 -0.448  0.230  0.797  0.169  1.061  0.297 -0.125  0.0   0.0   0.0   0.0   0.0
pc4 -0.209  0.165 -0.565  0.012  0.024 -0.253 -0.573 -0.056  0.0   0.0   0.0   0.0   0.0
pc5 -0.115 -0.412  0.136 -0.279 -0.279 -0.729  0.224 -0.343  0.0   0.0   0.0   0.0   0.0
pc6  0.160  0.375  0.352 -0.198  0.340  0.303  0.063  0.757  0.0   0.0   0.0   0.0   0.0


---
## Step 5b — Ground archetypes in real anchor artists instead of guessed audio values

Rather than trusting my hand-typed `AUDIO_INTENTS` numbers, use real, recognizable songs as ground truth: for each archetype, take a curated list of well-known artists, pull every matching song's *actual* Spotify-PC score from `song_pca.csv`, and average them into `Y_train`. This replaces a guess with a measurement, and as a side benefit keeps the reference points for each vibe anchored to familiar music rather than an arbitrary point in feature space.

**Relabeling note:** the artist list as drafted used category names (`pc1_energetic_club`, `pc2_acoustic_brunch`, `pc3_fine_dining_jazz`, `pc4_trendy_hipster`) that don't line up with this notebook's Yelp-PC numbering — e.g. "fine_dining_jazz" matches our `pc3` (fine dining) correctly, but "trendy_hipster" matches our `pc5` (hipster/trendy), not `pc4` (which we already corrected to mean loud/chaotic, not trendy). Remapped here:
- `acoustic_brunch` → `pc2` (brunch/breakfast)
- `fine_dining_jazz` → `pc3` (fine dining)
- `trendy_hipster` → `pc5` (hipster/trendy)
- `energetic_club` → `pc8` (good for groups) — closest real match; energetic/danceable/social fits "good for groups" better than any other archetype we have

**`pc1`, `pc4`, `pc6`, `pc7` didn't have anchors provided** — placeholder lists proposed below, flagged for you to review/replace since picking "the right famous dive-bar artist" is a domain judgment call, not something to default silently.

If an archetype has fewer than `MIN_ANCHOR_SONGS` real matches (e.g. a name doesn't appear in this dataset, or is spelled differently), it falls back to the Step 5 audio-intent projection instead of an unreliable few-song average.

In [24]:
import re

ANCHOR_ARTISTS = {
    # provided, remapped to the correct Yelp PC per the note above
    'pc2': ['Leon Bridges', 'Norah Jones', 'Jack Johnson', 'John Mayer', 'The Lumineers',
            'Maggie Rogers', 'Vance Joy', 'Kacey Musgraves', 'Fleet Foxes', 'Hozier'],
    'pc3': ['Miles Davis', 'John Coltrane', 'Frank Sinatra', 'Ella Fitzgerald', 'Bill Evans',
            'Sade', 'Diana Krall', 'Nat King Cole', 'Stan Getz', 'Nina Simone'],
    'pc5': ['Tame Impala', 'Khruangbin', 'Thundercat', 'Vampire Weekend', 'Mac DeMarco',
            'Arctic Monkeys', 'Phoebe Bridgers', 'LCD Soundsystem', 'The Strokes', 'Blood Orange'],
    'pc8': ['Daft Punk', 'Disclosure', 'Calvin Harris', 'The Weeknd', 'Dua Lipa',
            'Fred again..', 'RÜFÜS DU SOL', 'KAYTRANADA', 'Jamie xx', 'Peggy Gou'],
    # placeholders - not provided, review/replace these
    'pc1': ['Bruno Mars', 'John Legend', 'Michael Buble', 'Jason Mraz', 'Andra Day',
            'Alicia Keys', 'Adele', 'Ben Rector', 'Colbie Caillat', 'Gavin DeGraw'],
    'pc4': ['AC/DC', 'Guns N\' Roses', 'Red Hot Chili Peppers', 'Kings of Leon',
            'The White Stripes', 'Foo Fighters', 'Weezer', 'blink-182', 'Fall Out Boy', 'Panic! At The Disco'],
    'pc6': ['Tom Waits', 'The Black Keys', 'Alabama Shakes', 'Gary Clark Jr.', 'ZZ Top',
            'Jack White', 'The Rolling Stones', 'Bob Seger', 'Creedence Clearwater Revival', 'Whiskey Myers'],
    'pc7': ['FKA twigs', 'Billie Eilish', 'Frank Ocean', 'James Blake', 'Rhye',
            'Bon Iver', 'Daniel Caesar', 'SZA', 'Jorja Smith', 'Lianne La Havas'],
}

MIN_ANCHOR_SONGS = 5
Y_train_anchor = {}

for pc, artists in ANCHOR_ARTISTS.items():
    mask = pd.Series(False, index=songs.index)
    per_artist = {}
    for artist in artists:
        m = songs['artists'].str.contains(re.escape(artist), case=False, na=False)
        per_artist[artist] = int(m.sum())
        mask |= m
    n_matched = int(mask.sum())
    zero_hits = [a for a, c in per_artist.items() if c == 0]
    print(f"{pc}: {n_matched} matching songs" + (f"  (no matches for: {zero_hits})" if zero_hits else ""))
    Y_train_anchor[pc] = song_vecs[mask.values].mean(axis=0) if n_matched >= MIN_ANCHOR_SONGS else None

pc2: 183 matching songs  (no matches for: ['Vance Joy'])
pc3: 349 matching songs
pc5: 261 matching songs
pc8: 395 matching songs
pc1: 256 matching songs  (no matches for: ['Michael Buble'])
pc4: 279 matching songs  (no matches for: ["Guns N' Roses"])
pc6: 213 matching songs  (no matches for: ['The Rolling Stones', 'Creedence Clearwater Revival'])
pc7: 228 matching songs


In [25]:
Y_train = np.zeros((n_archetypes, N_SPOTIFY))
for row_i, pc in enumerate(ARCHETYPE_PCS):
    if Y_train_anchor.get(pc) is not None:
        Y_train[row_i] = Y_train_anchor[pc]
        source = 'real anchor songs'
    else:
        Y_train[row_i] = audio_to_spotify_pc(AUDIO_INTENTS[pc])
        source = 'hand-authored fallback (insufficient anchor coverage)'
    print(f"{pc}: target source = {source}")

model = LinearRegression(fit_intercept=False)
model.fit(X_train, Y_train)
W = model.coef_   # overwrites the Step 5 W with anchor-grounded targets

W_df = pd.DataFrame(W, index=SPOTIFY_PC_COLS, columns=YELP_PC_COLS)
print("\nWeight matrix W, rebuilt from anchor-grounded targets:")
print(W_df.round(3).to_string())

pc1: target source = real anchor songs
pc2: target source = real anchor songs
pc3: target source = real anchor songs
pc4: target source = real anchor songs
pc5: target source = real anchor songs
pc6: target source = real anchor songs
pc7: target source = real anchor songs
pc8: target source = real anchor songs

Weight matrix W, rebuilt from anchor-grounded targets:
       pc1    pc2    pc3    pc4    pc5    pc6    pc7    pc8  pc9  pc10  pc11  pc12  pc13
pc1  0.261 -0.266 -0.486  1.429  0.393  0.891 -0.406  1.400  0.0   0.0   0.0   0.0   0.0
pc2  0.321  0.205  0.201 -0.602  0.120  0.201  0.437  0.216  0.0   0.0   0.0   0.0   0.0
pc3  0.017 -0.175  0.541  0.193 -0.127  0.042  0.229 -0.395  0.0   0.0   0.0   0.0   0.0
pc4 -0.192 -0.098  0.123 -0.189 -0.059  0.013 -0.091 -0.302  0.0   0.0   0.0   0.0   0.0
pc5 -0.200 -0.462 -0.397 -0.417 -0.335 -0.241 -0.013  0.216  0.0   0.0   0.0   0.0   0.0
pc6 -0.556 -0.583  0.243 -0.251 -0.065 -0.193 -0.535  0.110  0.0   0.0   0.0   0.0   0.0


---
## Step 6 — Boss-level archetype evaluation (fixed diagnostics)

Same idea as notebook 05's Step 6: find the real restaurant that scores most extreme on each covered Yelp PC, project it through `W`, and check whether the recommended songs make sense — but with three fixes:

- **Normalize `y` before projecting.** `W`'s columns were calibrated against *unit-magnitude* one-hot archetypes, but real "boss" restaurants (the most extreme real example of each PC) have PC scores as large as 8–10 in magnitude (checked: `restaurant_pca.csv` pc1 max is 8.44, pc3 max is 10.29). Feeding that straight through `W` blows `s_hat` far outside the real song manifold — every archetype ends up recommending the same handful of extreme-outlier songs regardless of vibe. Normalizing `y` to unit norm first (same as the "FIX A" step from notebook 05) keeps `s_hat` on a scale comparable to what `W` was actually calibrated against.
- **Euclidean distance in standardized Spotify space** (song PCs have very different variances — pc1's std is ~1.8 vs pc6's ~0.86 — so distance needs standardizing first, same fix as the "FIXED MATH" cell in notebook 05).
- **The "dominant Spotify PC" diagnostic is computed from the same standardized vector used for ranking**, not a differently-scaled one — in notebook 05 those two were computed on different scales, so the printed "dominant PC" could disagree with what was actually driving the recommendations.

In [26]:
boss_restaurants = {}
for pc in ARCHETYPE_PCS:
    idx = restaurants[pc].idxmax()
    boss_restaurants[pc] = idx
    print(f"  {pc} ({YELP_PC_LABELS[pc]})\n    -> {restaurants.loc[idx, 'name']}  (score: {restaurants.loc[idx, pc]:.3f})")

scaler_song = StandardScaler()
song_vecs_scaled = scaler_song.fit_transform(song_vecs)

  pc1 (sit-down dinner (table service + casual + happy hour))
    -> The BAO  (score: 8.436)
  pc2 (brunch / breakfast)
    -> Living Room Coffee & Kitchen  (score: 6.836)
  pc3 (fine dining (upscale + classy + romantic, not casual))
    -> 1799 Kitchen and Cocktails  (score: 10.294)
  pc4 (loud / chaotic (high noise, lower stars))
    -> Effervescence  (score: 6.751)
  pc5 (hipster / trendy)
    -> Bok Bar  (score: 7.208)
  pc6 (dive bar (very divey, some late-night overlap))
    -> Verti Marte  (score: 7.382)
  pc7 (late-night)
    -> Barcelona Wine Bar  (score: 6.590)
  pc8 (good for groups (divey/hipster-leaning, no table service))
    -> Strange Bird  (score: 5.157)


In [27]:
print("=== BOSS-LEVEL MATCHUP EVALUATION (v2) ===\n")

for pc in ARCHETYPE_PCS:
    rest_idx  = boss_restaurants[pc]
    rest_name = restaurants.loc[rest_idx, 'name']
    y = restaurant_vecs[rest_idx]
    y_normalized = y / (np.linalg.norm(y) + 1e-9)

    s_hat        = W @ y_normalized
    s_hat_scaled = scaler_song.transform(s_hat.reshape(1, -1))

    dominant_spc = SPOTIFY_PC_COLS[np.argmax(np.abs(s_hat_scaled[0]))]
    dominant_val = s_hat_scaled[0][np.argmax(np.abs(s_hat_scaled[0]))]

    distances = pairwise_distances(s_hat_scaled, song_vecs_scaled, metric='euclidean')[0]
    top_idx   = np.argsort(distances)[:3]

    print(f"{'-'*65}")
    print(f"  Restaurant ({pc} - {YELP_PC_LABELS[pc]})")
    print(f"  -> {rest_name}")
    print(f"  Dominant Spotify PC (scaled): {dominant_spc} ({SPOTIFY_PC_LABELS[dominant_spc]})  val={dominant_val:.3f}")
    print(f"  Top 3 recommended songs:")
    for rank, i in enumerate(top_idx, 1):
        print(f"    {rank}. \"{songs.loc[i,'name']}\" by {songs.loc[i,'artists']}  (dist={distances[i]:.3f})")
    print()

=== BOSS-LEVEL MATCHUP EVALUATION (v2) ===

-----------------------------------------------------------------
  Restaurant (pc1 - sit-down dinner (table service + casual + happy hour))
  -> The BAO
  Dominant Spotify PC (scaled): pc5 (spoken word / pure instrumental)  val=-0.494
  Top 3 recommended songs:
    1. "D-Milk (Nine Ounce Nails Mix)" by ['Andy Colvin']  (dist=0.217)
    2. "D-Milk (Live @ Kcmu 1989)" by ['Andy Colvin', 'Ed Hall']  (dist=0.217)
    3. "Dead Bitch" by ['Evil Pimp']  (dist=0.226)

-----------------------------------------------------------------
  Restaurant (pc2 - brunch / breakfast)
  -> Living Room Coffee & Kitchen
  Dominant Spotify PC (scaled): pc5 (spoken word / pure instrumental)  val=-0.698
  Top 3 recommended songs:
    1. "Cry Cry Cry" by ['Ty Segall']  (dist=0.194)
    2. "Feels Like Heaven" by ['Ulrich Ellison and Tribe']  (dist=0.206)
    3. "The Little Lie That You Told" by ['Zettaimu']  (dist=0.230)

-----------------------------------------------

---
## Step 7 — Detect and penalize hub songs (population-wide)

The boss-level test above only checks 8 restaurants. The real hub question is: across *all* ~34,800 restaurants, do a handful of songs dominate the top-10 for a disproportionate, dissimilar share of them? Six dimensions is cheap enough to check exactly with a KD-tree rather than guessing from a small sample.

1. Project every restaurant through `W` (same normalize -> scale steps as the boss-level eval).
2. Use `NearestNeighbors` to find each restaurant's real top-10 nearest songs.
3. Count each song's in-degree — how many restaurants have it in their top-10.
4. Flag songs whose in-degree exceeds 5% of restaurants as hubs.

In [28]:
from sklearn.neighbors import NearestNeighbors
from collections import Counter

restaurant_vecs_normalized = restaurant_vecs / (np.linalg.norm(restaurant_vecs, axis=1, keepdims=True) + 1e-9)
s_hat_all        = restaurant_vecs_normalized @ W.T
s_hat_all_scaled = scaler_song.transform(s_hat_all)

nn = NearestNeighbors(n_neighbors=10, algorithm='auto', metric='euclidean')
nn.fit(song_vecs_scaled)
_, top10_idx = nn.kneighbors(s_hat_all_scaled)

n_restaurants  = len(restaurants)
hub_threshold  = 0.05 * n_restaurants
song_in_degree = Counter(top10_idx.flatten())

hub_songs = {i: c for i, c in song_in_degree.items() if c > hub_threshold}
print(f"{n_restaurants} restaurants, hub threshold = {hub_threshold:.0f} restaurants (5%)")
print(f"{len(hub_songs)} songs flagged as hubs out of {len(song_in_degree)} songs that appear in any top-10\n")

print("Worst offenders:")
worst = sorted(hub_songs.items(), key=lambda kv: -kv[1])[:10]
for song_idx, count in worst:
    pct = count / n_restaurants
    print(f"  \"{songs.loc[song_idx,'name']}\" by {songs.loc[song_idx,'artists']}"
          f"  -> top-10 for {count} restaurants ({pct:.1%})")

34808 restaurants, hub threshold = 1740 restaurants (5%)
46 songs flagged as hubs out of 1580 songs that appear in any top-10

Worst offenders:
  "I'll remember you" by ['Duskus']  -> top-10 for 6449 restaurants (18.5%)
  "Swirling Skies" by ['Peter Pearson']  -> top-10 for 4657 restaurants (13.4%)
  "Run It" by ['Project Paradis', 'Mr. Carmack', 'Promnite']  -> top-10 for 4289 restaurants (12.3%)
  "We're Remembering Right Now" by ['Koss', 'Henriksson', 'Mullaert']  -> top-10 for 3930 restaurants (11.3%)
  "Fox's Sunday Best" by ['Naomi Randall', 'Tom Gaskell']  -> top-10 for 3751 restaurants (10.8%)
  "Intro" by ['M.M.M.F.D.']  -> top-10 for 3726 restaurants (10.7%)
  "Human Om" by ['TOBACCO']  -> top-10 for 3624 restaurants (10.4%)
  "Boot Up" by ["Lord D'Andre"]  -> top-10 for 3588 restaurants (10.3%)
  "VIVID DREAMS" by ['KAYTRANADA', 'River Tiber']  -> top-10 for 3575 restaurants (10.3%)
  "The Only Shrine I've Seen" by ['DARKSIDE']  -> top-10 for 3553 restaurants (10.2%)


In [29]:
in_degree_arr = np.zeros(len(songs))
for song_idx, count in song_in_degree.items():
    in_degree_arr[song_idx] = count

penalty = np.ones(len(songs))
over = in_degree_arr > hub_threshold
penalty[over] = np.sqrt(in_degree_arr[over] / hub_threshold)   # only hub songs get penalized, and only in proportion to how far over the threshold they are

print("=== BOSS-LEVEL MATCHUP: BEFORE vs AFTER HUB PENALTY ===\n")

for pc in ARCHETYPE_PCS:
    rest_idx = boss_restaurants[pc]
    y = restaurant_vecs[rest_idx]
    y_normalized = y / (np.linalg.norm(y) + 1e-9)
    s_hat        = W @ y_normalized
    s_hat_scaled = scaler_song.transform(s_hat.reshape(1, -1))

    distances          = pairwise_distances(s_hat_scaled, song_vecs_scaled, metric='euclidean')[0]
    distances_penalized = distances * penalty

    top_before = np.argsort(distances)[:3]
    top_after  = np.argsort(distances_penalized)[:3]

    print(f"{'-'*65}")
    print(f"  Restaurant ({pc} - {YELP_PC_LABELS[pc]}) -> {restaurants.loc[rest_idx, 'name']}")
    print(f"  BEFORE penalty:")
    for rank, i in enumerate(top_before, 1):
        flag = "  [HUB]" if in_degree_arr[i] > hub_threshold else ""
        print(f"    {rank}. \"{songs.loc[i,'name']}\" by {songs.loc[i,'artists']}  (dist={distances[i]:.3f}){flag}")
    print(f"  AFTER penalty:")
    for rank, i in enumerate(top_after, 1):
        flag = "  [HUB]" if in_degree_arr[i] > hub_threshold else ""
        print(f"    {rank}. \"{songs.loc[i,'name']}\" by {songs.loc[i,'artists']}  (dist={distances_penalized[i]:.3f}){flag}")
    print()

=== BOSS-LEVEL MATCHUP: BEFORE vs AFTER HUB PENALTY ===

-----------------------------------------------------------------
  Restaurant (pc1 - sit-down dinner (table service + casual + happy hour)) -> The BAO
  BEFORE penalty:
    1. "D-Milk (Nine Ounce Nails Mix)" by ['Andy Colvin']  (dist=0.217)
    2. "D-Milk (Live @ Kcmu 1989)" by ['Andy Colvin', 'Ed Hall']  (dist=0.217)
    3. "Dead Bitch" by ['Evil Pimp']  (dist=0.226)
  AFTER penalty:
    1. "D-Milk (Live @ Kcmu 1989)" by ['Andy Colvin', 'Ed Hall']  (dist=0.217)
    2. "D-Milk (Nine Ounce Nails Mix)" by ['Andy Colvin']  (dist=0.217)
    3. "Dead Bitch" by ['Evil Pimp']  (dist=0.226)

-----------------------------------------------------------------
  Restaurant (pc2 - brunch / breakfast) -> Living Room Coffee & Kitchen
  BEFORE penalty:
    1. "Cry Cry Cry" by ['Ty Segall']  (dist=0.194)
    2. "Feels Like Heaven" by ['Ulrich Ellison and Tribe']  (dist=0.206)
    3. "The Little Lie That You Told" by ['Zettaimu']  (dist=0.230)
  

---
## Step 7b — Validate the penalty on typical (non-extreme) restaurants

The 8 "boss" restaurants above are each the single most extreme real example of one Yelp PC — outliers by construction. None of their top-3 songs happened to be flagged hubs, so the before/after comparison above doesn't actually show the penalty doing anything. That's a property of testing on outliers, not evidence the penalty is inert. Checking against a random sample of ordinary restaurants instead, where hub songs are more likely to actually show up in the unpenalized results.

In [30]:
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(restaurants), size=15, replace=False)

n_changed = 0
for rest_idx in sample_idx:
    y = restaurant_vecs[rest_idx]
    y_normalized = y / (np.linalg.norm(y) + 1e-9)
    s_hat        = W @ y_normalized
    s_hat_scaled = scaler_song.transform(s_hat.reshape(1, -1))

    distances           = pairwise_distances(s_hat_scaled, song_vecs_scaled, metric='euclidean')[0]
    distances_penalized = distances * penalty

    top_before = np.argsort(distances)[:3]
    top_after  = np.argsort(distances_penalized)[:3]
    changed = not np.array_equal(top_before, top_after)
    n_changed += changed

    print(f"{'-'*65}")
    print(f"  {restaurants.loc[rest_idx, 'name']}" + ("  <- penalty changed the top-3" if changed else ""))
    for rank, i in enumerate(top_before, 1):
        flag = "  [HUB]" if in_degree_arr[i] > hub_threshold else ""
        print(f"    {rank}. \"{songs.loc[i,'name']}\" by {songs.loc[i,'artists']}  (dist={distances[i]:.3f}){flag}")

print(f"\n{n_changed} / {len(sample_idx)} sampled restaurants had their top-3 changed by the hub penalty")

-----------------------------------------------------------------
  Godfather's Pizza  <- penalty changed the top-3
    1. "Boot Up" by ["Lord D'Andre"]  (dist=0.307)  [HUB]
    2. "Anagami" by ['David S. Ware', 'William Parker', 'Warren Smith']  (dist=0.393)  [HUB]
    3. "The Ceiling Is Bendin' - 2018 Remaster" by ['The Flaming Lips']  (dist=0.435)  [HUB]
-----------------------------------------------------------------
  The Old Spaghetti Factory
    1. "Fred Goes Out At Night" by ['The Kropotkins']  (dist=0.226)  [HUB]
    2. "Priceless" by ['Anfa Rose']  (dist=0.415)
    3. "Dynamic Duo" by ['The Lewis Connection']  (dist=0.465)
-----------------------------------------------------------------
  Pomme Restaurant  <- penalty changed the top-3
    1. "I'll remember you" by ['Duskus']  (dist=0.292)  [HUB]
    2. "Turn off the Lights" by ['Steve Wingfield']  (dist=0.366)  [HUB]
    3. "Stumble" by ['Peacehead']  (dist=0.463)  [HUB]
-----------------------------------------------------

---
## Step 8 — Try it: a random, feature-rich restaurant

Picking a restaurant at random isn't useful if it lands on one with almost every vibe flag at 0 — the recommendation would just reflect noise/defaults rather than an actual profile. Filter to restaurants with at least `MIN_FEATURES` vibe flags set (Ambience.*, HasTV, HappyHour, RestaurantsGoodForGroups, GoodForMeal.*, RestaurantsTableService), then pick randomly among those and run the full pipeline: normalize -> project through `W` -> scale -> hub-penalized nearest neighbors.

In [31]:
BINARY_VIBE_COLS = [c for c in YELP_FEATURES if c not in ('NoiseLevel', 'stars')]

yelp_raw = pd.read_csv(os.path.join(PROCESSED, 'yelp_ml_ready.csv'))
yelp_raw['feature_richness'] = yelp_raw[BINARY_VIBE_COLS].sum(axis=1)

MIN_FEATURES = 4
rich_ids   = set(yelp_raw.loc[yelp_raw['feature_richness'] >= MIN_FEATURES, 'business_id'])
candidates = restaurants[restaurants['business_id'].isin(rich_ids)]

print(f"{len(candidates)} / {len(restaurants)} restaurants have >= {MIN_FEATURES} vibe features set")

rng = np.random.default_rng()
test_idx = candidates.sample(1, random_state=rng.integers(0, 1_000_000)).index[0]
test_row = restaurants.loc[test_idx]

print(f"\nTesting restaurant: {test_row['name']}")

profile = yelp_raw.loc[yelp_raw['business_id'] == test_row['business_id'], BINARY_VIBE_COLS + ['NoiseLevel', 'stars']]
print("\nRaw feature profile:")
print(profile.T.rename(columns={profile.index[0]: 'value'}).to_string())

16257 / 34808 restaurants have >= 4 vibe features set

Testing restaurant: La Mia Toscana

Raw feature profile:
                          value
Ambience.romantic           0.0
Ambience.divey              0.0
Ambience.classy             1.0
Ambience.hipster            0.0
Ambience.trendy             0.0
Ambience.upscale            0.0
Ambience.casual             0.0
HasTV                       1.0
HappyHour                   1.0
RestaurantsGoodForGroups    1.0
GoodForMeal.breakfast       0.0
GoodForMeal.brunch          0.0
GoodForMeal.latenight       0.0
GoodForMeal.dinner          0.0
RestaurantsTableService     0.0
NoiseLevel                  1.0
stars                       3.0


In [32]:
y            = restaurant_vecs[test_idx]
y_normalized = y / (np.linalg.norm(y) + 1e-9)
s_hat        = W @ y_normalized
s_hat_scaled = scaler_song.transform(s_hat.reshape(1, -1))

distances           = pairwise_distances(s_hat_scaled, song_vecs_scaled, metric='euclidean')[0]
distances_penalized = distances * penalty
top5 = np.argsort(distances_penalized)[:5]

dominant_spc = SPOTIFY_PC_COLS[np.argmax(np.abs(s_hat_scaled[0]))]
print(f"Dominant Spotify PC: {dominant_spc} ({SPOTIFY_PC_LABELS[dominant_spc]})")

print(f"\nTop 5 recommended songs for \"{test_row['name']}\":")
for rank, i in enumerate(top5, 1):
    flag = "  [HUB]" if in_degree_arr[i] > hub_threshold else ""
    print(f"  {rank}. \"{songs.loc[i,'name']}\" by {songs.loc[i,'artists']}  (dist={distances_penalized[i]:.3f}){flag}")

Dominant Spotify PC: pc2 (happy + danceable (acoustic-leaning))

Top 5 recommended songs for "La Mia Toscana":
  1. "Call The Preacher" by ['Jonathon Boogie Long']  (dist=0.331)
  2. "Slide (Slidin' the Blues)" by ['The Stooges']  (dist=0.341)
  3. "Into The Viper's Den" by ['Bassline Drift']  (dist=0.384)  [HUB]
  4. "Riff Unknown" by ['David S. Ware Quartet']  (dist=0.408)
  5. "Love Is Here and Now You're Gone" by ['Lambchop']  (dist=0.415)


---
## Step 9 — Explainability: raw feature weights

Two things worth separating: the **raw restaurant features** (already shown above — the actual `yelp_ml_ready.csv` values for this restaurant), and the **weights** that determine how much each raw feature actually matters to the recommendation.

There are two weight matrices in this pipeline, chained together:
1. `yelp_loadings` (Step 2/3) — how each raw Yelp feature contributes to each of the 13 Yelp PCs. This came straight out of `pca_yelp.components_`, not authored by hand.
2. `W` — how each Yelp PC maps to each Spotify PC. This one *is* hand-authored/anchor-grounded (Steps 5/5b) — the part of the pipeline that's still a hypothesis, not a fitted model.

Multiplying them (`M = W @ yelp_loadings`) collapses the chain into a single **raw Yelp feature -> Spotify PC** weight matrix — this shows, directly, how much each individual attribute (e.g. `Ambience.classy`, `GoodForMeal.brunch`) pushes toward each Spotify PC, skipping the intermediate PCA representation.

In [33]:
V = pca_yelp.components_          # (13 Yelp PCs, 17 raw features)
M = W @ V                          # (6 Spotify PCs, 17 raw features) - direct raw-feature weights

M_df = pd.DataFrame(M, index=SPOTIFY_PC_COLS, columns=YELP_FEATURES)
print("Effective weight of each raw Yelp feature on each Spotify PC (W @ yelp_loadings):")
print(M_df.round(3).to_string())

Effective weight of each raw Yelp feature on each Spotify PC (W @ yelp_loadings):
     Ambience.romantic  Ambience.divey  Ambience.classy  Ambience.hipster  Ambience.trendy  Ambience.upscale  Ambience.casual  HasTV  HappyHour  RestaurantsGoodForGroups  GoodForMeal.breakfast  GoodForMeal.brunch  GoodForMeal.latenight  GoodForMeal.dinner  RestaurantsTableService  NoiseLevel  stars
pc1              0.228           1.666           -0.174             0.561            0.322             0.361           -0.275  0.805      0.298                     0.789                  0.084               0.075                  0.449              -0.238                   -0.245       0.453 -0.340
pc2              0.189          -0.028            0.014             0.132           -0.005             0.026            0.381 -0.339     -0.194                     0.116                  0.006              -0.008                  0.191               0.298                    0.005      -0.368  0.461
pc3              0

In [34]:
scaler_yelp = joblib.load(os.path.join(PROCESSED, 'yelp_scaler.pkl'))

raw_row = yelp_raw.loc[yelp_raw['business_id'] == test_row['business_id'], YELP_FEATURES]
raw_row = raw_row.fillna(yelp_raw[YELP_FEATURES].median())   # guard against NoiseLevel nulls
x_scaled = scaler_yelp.transform(raw_row)[0]

dominant_row  = M[SPOTIFY_PC_COLS.index(dominant_spc)]
contributions = dominant_row * x_scaled
contrib_series = pd.Series(contributions, index=YELP_FEATURES).reindex(
    pd.Series(contributions, index=YELP_FEATURES).abs().sort_values(ascending=False).index
)

print(f"Which raw features of \"{test_row['name']}\" actually drove {dominant_spc} ({SPOTIFY_PC_LABELS[dominant_spc]})?")
print("(weight x this restaurant's scaled feature value = contribution; sorted by |contribution|)\n")
print(contrib_series.round(3).to_string())

Which raw features of "La Mia Toscana" actually drove pc2 (happy + danceable (acoustic-leaning))?
(weight x this restaurant's scaled feature value = contribution; sorted by |contribution|)

Ambience.casual            -0.417
HappyHour                  -0.380
stars                      -0.322
GoodForMeal.dinner         -0.226
HasTV                      -0.209
NoiseLevel                 -0.055
RestaurantsGoodForGroups    0.049
GoodForMeal.latenight      -0.038
Ambience.classy             0.035
Ambience.romantic          -0.025
Ambience.hipster           -0.021
Ambience.divey              0.005
RestaurantsTableService    -0.003
Ambience.upscale           -0.003
GoodForMeal.brunch          0.002
GoodForMeal.breakfast      -0.002
Ambience.trendy             0.001
